# Uber Data Analysis with PySpark RDD

This notebook analyzes Uber trip data using RDD API with focus on partitioning and performance optimization.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName('Uber Data Analysis') \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark Version: {spark.version}")
print(f"Default Parallelism: {sc.defaultParallelism}")

## Dataset Information

Uber trip data typically contains:
- Trip date/time
- Pickup/dropoff locations (lat/long)
- Base code (dispatch base)
- Trip details

We'll analyze patterns in trip data using RDD operations.

## Load Data and Check Partitions

In [ ]:
# Load Uber data
uber_rdd = sc.textFile("/content/sample_data/uber_data.csv")

# Check initial partitions
print(f"Number of partitions: {uber_rdd.getNumPartitions()}")
print(f"Total records: {uber_rdd.count()}")

# Show sample data
print("\nSample data (first 5 lines):")
for line in uber_rdd.take(5):
    print(line)

## Understanding Partitioning

In [ ]:
# Function to check records per partition
def count_per_partition(iterator):
    yield sum(1 for _ in iterator)

partition_counts = uber_rdd.mapPartitions(count_per_partition).collect()

print("Records per partition:")
for i, count in enumerate(partition_counts):
    print(f"Partition {i}: {count} records")

print(f"\nTotal: {sum(partition_counts)} records")

## Parse Data

In [ ]:
# Remove header
header = uber_rdd.first()
uber_data = uber_rdd.filter(lambda line: line != header)

print(f"Header: {header}")
print(f"\nData records: {uber_data.count()}")

In [ ]:
# Parse CSV lines
def parse_uber_record(line):
    """Parse CSV line into structured format"""
    try:
        fields = line.split(',')
        return {
            'datetime': fields[0],
            'lat': float(fields[1]),
            'lon': float(fields[2]),
            'base': fields[3]
        }
    except (ValueError, IndexError):
        return None

# Parse and filter invalid records
parsed_uber = uber_data.map(parse_uber_record).filter(lambda x: x is not None)

print("Parsed records sample:")
for record in parsed_uber.take(5):
    print(record)

## Analysis: Trips by Base

In [ ]:
# Count trips per base
trips_by_base = parsed_uber \
    .map(lambda x: (x['base'], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print("Trips by Base:")
for base, count in trips_by_base.collect():
    print(f"{base}: {count:,} trips")

## Analysis: Trips by Date

In [ ]:
# Extract date from datetime and count
def extract_date(record):
    """Extract date from datetime string"""
    datetime_str = record['datetime']
    # Assuming format: 'M/D/YYYY H:MM:SS'
    date = datetime_str.split()[0]
    return date

trips_by_date = parsed_uber \
    .map(lambda x: (extract_date(x), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(10)

print("Top 10 Dates by Trip Count:")
for date, count in trips_by_date:
    print(f"{date}: {count:,} trips")

## Analysis: Trips by Hour

In [ ]:
# Extract hour from datetime
def extract_hour(record):
    """Extract hour from datetime string"""
    datetime_str = record['datetime']
    # Extract time part and get hour
    time_part = datetime_str.split()[1]  # Get 'H:MM:SS'
    hour = int(time_part.split(':')[0])  # Get hour
    return hour

trips_by_hour = parsed_uber \
    .map(lambda x: (extract_hour(x), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[0])

print("Trips by Hour of Day:")
for hour, count in trips_by_hour.collect():
    bar = '█' * (count // 1000)
    print(f"{hour:2d}:00 - {count:6,} trips {bar}")

## Repartitioning Analysis

### Compare Performance: Different Partition Counts

In [ ]:
# Test with different partition counts
def test_partition_performance(rdd, num_partitions):
    """Test aggregation performance with different partition counts"""
    start = time.time()
    
    # Repartition
    repartitioned = rdd.repartition(num_partitions)
    
    # Perform aggregation
    result = repartitioned \
        .map(lambda x: (x['base'], 1)) \
        .reduceByKey(lambda a, b: a + b) \
        .collect()
    
    elapsed = time.time() - start
    return elapsed, result

# Test different partition counts
partition_sizes = [2, 4, 8, 16]
results = {}

print("Testing different partition counts...\n")
for num_parts in partition_sizes:
    elapsed, _ = test_partition_performance(parsed_uber, num_parts)
    results[num_parts] = elapsed
    print(f"{num_parts} partitions: {elapsed:.4f} seconds")

# Find optimal
optimal = min(results, key=results.get)
print(f"\nOptimal partition count: {optimal} ({results[optimal]:.4f} seconds)")

### Coalesce vs Repartition

In [ ]:
# Compare coalesce vs repartition
print(f"Original partitions: {parsed_uber.getNumPartitions()}")

# Using coalesce (decreases partitions without full shuffle)
start = time.time()
coalesced = parsed_uber.coalesce(4)
coalesced.count()  # Action to trigger execution
coalesce_time = time.time() - start
print(f"\nCoalesce to 4 partitions: {coalesce_time:.4f} seconds")
print(f"Resulting partitions: {coalesced.getNumPartitions()}")

# Using repartition (full shuffle)
start = time.time()
repartitioned = parsed_uber.repartition(4)
repartitioned.count()  # Action to trigger execution
repartition_time = time.time() - start
print(f"\nRepartition to 4 partitions: {repartition_time:.4f} seconds")
print(f"Resulting partitions: {repartitioned.getNumPartitions()}")

print(f"\nCoalesce is {repartition_time/coalesce_time:.2f}x faster (no shuffle needed)")

## Advanced: Custom Partitioning

In [ ]:
# Custom partitioner based on base code
def base_partitioner(key):
    """Hash partitioner based on base code"""
    return hash(key) % 4

# Create key-value pairs and partition by base
base_partitioned = parsed_uber \
    .map(lambda x: (x['base'], x)) \
    .partitionBy(4, base_partitioner)

print(f"Custom partitioned RDD has {base_partitioned.getNumPartitions()} partitions")

# Check distribution
def count_by_partition_and_base(iterator):
    """Count records per base in each partition"""
    from collections import Counter
    counter = Counter()
    for base, record in iterator:
        counter[base] += 1
    return iter(counter.items())

partition_dist = base_partitioned \
    .mapPartitionsWithIndex(
        lambda idx, it: [(idx, base, count) for base, count in count_by_partition_and_base(it)]
    ) \
    .collect()

print("\nDistribution of bases across partitions:")
for partition_id, base, count in sorted(partition_dist):
    print(f"Partition {partition_id}, Base {base}: {count} records")

## Caching for Iterative Operations

In [ ]:
# Compare performance with and without caching
test_rdd = parsed_uber.sample(False, 0.5)  # 50% sample

# Without caching
start = time.time()
count1 = test_rdd.count()
count2 = test_rdd.count()
count3 = test_rdd.count()
time_no_cache = time.time() - start

print(f"3 counts without cache: {time_no_cache:.4f} seconds")

# With caching
cached_rdd = test_rdd.cache()
start = time.time()
count1 = cached_rdd.count()  # First count loads into cache
count2 = cached_rdd.count()  # Subsequent counts use cache
count3 = cached_rdd.count()
time_with_cache = time.time() - start

print(f"3 counts with cache: {time_with_cache:.4f} seconds")
print(f"\nSpeedup: {time_no_cache/time_with_cache:.2f}x")

# Unpersist to free memory
cached_rdd.unpersist()

## Summary Statistics

In [ ]:
# Calculate comprehensive statistics
total_trips = parsed_uber.count()
unique_bases = parsed_uber.map(lambda x: x['base']).distinct().count()

# Average trips per base
avg_trips_per_base = total_trips / unique_bases

# Busiest hour
busiest_hour, max_trips = trips_by_hour \
    .sortBy(lambda x: x[1], ascending=False) \
    .first()

print("\n" + "="*50)
print("UBER DATA SUMMARY")
print("="*50)
print(f"Total Trips: {total_trips:,}")
print(f"Unique Bases: {unique_bases}")
print(f"Average Trips per Base: {avg_trips_per_base:,.0f}")
print(f"Busiest Hour: {busiest_hour}:00 ({max_trips:,} trips)")
print("="*50)

## Key Learnings

### Partitioning:
- **Default partitioning**: Based on file size and block size
- **repartition()**: Full shuffle, increases or decreases partitions
- **coalesce()**: No shuffle, only decreases partitions (faster)
- **Custom partitioner**: Control data distribution based on keys

### Performance:
- Too few partitions: Underutilizes cluster
- Too many partitions: Overhead from task scheduling
- Optimal: Usually 2-4 partitions per CPU core

### Caching:
- Use `.cache()` or `.persist()` for iterative operations
- Significant speedup when RDD is used multiple times
- Remember to `.unpersist()` when done

In [ ]:
# Stop Spark Session
spark.stop()